# 04. Named Entity Recognition with spaCy for Practical Information Extraction


## 1. What you will build

- A document entity extraction pipeline using spaCy NER.
- A structured evaluation against gold entities stored in external data files.
- A reusable helper to extract and review entities at document level.


## 2. When to use this in real companies

Use this approach when teams need structured fields from unstructured text: contracts, emails, support notes, or reports. It is useful for analytics enrichment, rule automation, and downstream validation workflows.


## 3. Business goal

Extract `PERSON`, `ORG`, `GPE`, `DATE`, and `MONEY` entities from business text and measure extraction quality against reference annotations.


<h2 style="color:#f8fafc; font-family:Arial, sans-serif; margin-bottom: 16px;">
  NER with spaCy vs NER with Fine-Tuned BERT
</h2>

<table style="width:100%; border-collapse:collapse; font-family:Arial, sans-serif; background:#111827; color:#e5e7eb;">

  <tr style="background:#2563eb; color:white;">
    <th style="padding:12px; border:1px solid #374151; text-align:left;">Aspect</th>
    <th style="padding:12px; border:1px solid #374151; text-align:left;">NER with spaCy</th>
    <th style="padding:12px; border:1px solid #374151; text-align:left;">NER with Fine-Tuned BERT</th>
  </tr>

  <tr>
    <td style="padding:12px; border:1px solid #374151;"><strong>How it starts</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Uses a pre-trained NER pipeline (e.g., <code>en_core_web_sm</code>).</td>
    <td style="padding:12px; border:1px solid #374151;">Starts from a pre-trained language model (e.g., <code>distilbert-base-cased</code>) plus a new token-classification head.</td>
  </tr>

  <tr style="background:#1f2937;">
    <td style="padding:12px; border:1px solid #374151;"><strong>How it learns NER labels</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Already learned generic labels in pretraining package.</td>
    <td style="padding:12px; border:1px solid #374151;">Learns your labels during fine-tuning from BIO tags (<code>B-PER</code>, <code>I-ORG</code>, etc.) and <code>label2id/id2label</code>.</td>
  </tr>

  <tr>
    <td style="padding:12px; border:1px solid #374151;"><strong>Need labeled data?</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Not required for zero-shot usage.</td>
    <td style="padding:12px; border:1px solid #374151;">Yes, required for good task/domain performance.</td>
  </tr>

  <tr style="background:#1f2937;">
    <td style="padding:12px; border:1px solid #374151;"><strong>Do you need all names in the world?</strong></td>
    <td style="padding:12px; border:1px solid #374151;">No.</td>
    <td style="padding:12px; border:1px solid #374151;">No. You need representative domain examples, not every possible name.</td>
  </tr>

  <tr>
    <td style="padding:12px; border:1px solid #374151;"><strong>Context understanding</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Good for common/general patterns.</td>
    <td style="padding:12px; border:1px solid #374151;">Stronger contextual understanding and better adaptation to internal language.</td>
  </tr>

  <tr style="background:#1f2937;">
    <td style="padding:12px; border:1px solid #374151;"><strong>Inference speed</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Fast (CPU-friendly).</td>
    <td style="padding:12px; border:1px solid #374151;">Slower than spaCy (benefits from GPU).</td>
  </tr>

  <tr>
    <td style="padding:12px; border:1px solid #374151;"><strong>Computational cost</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Low.</td>
    <td style="padding:12px; border:1px solid #374151;">Higher training/inference cost.</td>
  </tr>

  <tr style="background:#1f2937;">
    <td style="padding:12px; border:1px solid #374151;"><strong>Best use case</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Quick baseline and production with strict latency/cost limits.</td>
    <td style="padding:12px; border:1px solid #374151;">Business-critical extraction quality and domain-specific entity language.</td>
  </tr>

  <tr>
    <td style="padding:12px; border:1px solid #374151;"><strong>Output</strong></td>
    <td style="padding:12px; border:1px solid #374151;">Entity spans + labels from spaCy pipeline.</td>
    <td style="padding:12px; border:1px solid #374151;">Token BIO predictions aggregated to entity spans.</td>
  </tr>

</table>

<p style="margin-top:16px; color:#9ca3af; font-family:Arial;">
  In short: spaCy is ideal for speed and simplicity; fine-tuned BERT is ideal when you need higher domain-specific NER quality and can afford labeled data + training cost.
</p>

## 4. Imports and setup


In [1]:
from pathlib import Path

import pandas as pd
import spacy

In [2]:
try:
    nlp = spacy.load("en_core_web_sm")
except OSError as exc:
    raise OSError("Run: python -m spacy download en_core_web_sm") from exc

## 5. Load datasets from `data/04_data`


In [3]:
DOCS_PATH = Path("../data/04_data/ner_documents.csv")
GOLD_PATH = Path("../data/04_data/ner_gold_entities.csv")

docs_df = pd.read_csv(DOCS_PATH)
gold_df = pd.read_csv(GOLD_PATH)

print(f"Documents: {len(docs_df):,}")
print(f"Gold entities: {len(gold_df):,}")
docs_df.head()

Documents: 160
Gold entities: 800


,doc_id,text
0,NER4-0001,"On 2026-01-12, Liam Walker from Fabrikam Ops s..."
1,NER4-0002,"On 2026-08-05, Emma Rossi from Apex Systems si..."
2,NER4-0003,"On 2026-05-09, Liam Walker from Nova Retail si..."
3,NER4-0004,"On 2026-03-18, Ana Ruiz from Helios Energy sig..."
4,NER4-0005,"On 2026-07-21, John Miller from Orion Logistic..."


In [4]:
gold_df.head()

,doc_id,entity_text,label
0,NER4-0001,2026-01-12,DATE
1,NER4-0001,Liam Walker,PERSON
2,NER4-0001,Fabrikam Ops,ORG
3,NER4-0001,Lisbon,GPE
4,NER4-0001,"$18,900",MONEY


## 6. Entity extraction function


In [5]:
def extract_entities(text: str):
    """Extract entities as (text, label) pairs with spaCy."""
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]


rows = []
for _, r in docs_df.iterrows():
    for ent_text, ent_label in extract_entities(r["text"]):
        rows.append({"doc_id": r["doc_id"], "entity_text": ent_text, "label": ent_label})

pred_df = pd.DataFrame(rows)
pred_df.head(10)

,doc_id,entity_text,label
0,NER4-0001,2026-01-12,DATE
1,NER4-0001,Liam Walker,PERSON
2,NER4-0001,Fabrikam Ops,ORG
3,NER4-0001,Lisbon,GPE
4,NER4-0001,"18,900",MONEY
5,NER4-0002,2026-08-05,DATE
6,NER4-0002,Emma Rossi,PERSON
7,NER4-0002,Apex Systems,ORG
8,NER4-0002,New York,GPE
9,NER4-0002,"4,850",MONEY


In [6]:
pred_df["label"].value_counts().sort_index()

label
DATE      160
GPE       165
LOC        13
MONEY     160
ORG       118
PERSON    158
Name: count, dtype: int64

## 7. Evaluation against gold entities

Exact match on `(doc_id, entity_text, label)`.


In [7]:
def to_set(frame: pd.DataFrame):
    return set((row.doc_id, row.entity_text, row.label) for row in frame.itertuples(index=False))

gold_set = to_set(gold_df)
pred_set = to_set(pred_df)

tp = len(gold_set & pred_set)
fp = len(pred_set - gold_set)
fn = len(gold_set - pred_set)

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

pd.Series({
    "true_positives": tp,
    "false_positives": fp,
    "false_negatives": fn,
    "precision": precision,
    "recall": recall,
    "f1": f1,
}).round(3)

true_positives     572.000
false_positives    202.000
false_negatives    228.000
precision            0.739
recall               0.715
f1                   0.727
dtype: float64

## 8. Document-level inspection helper


In [8]:
def review_document(doc_id: str):
    """Return one document with its predicted and gold entities."""
    text = docs_df.loc[docs_df["doc_id"] == doc_id, "text"].iloc[0]
    pred = pred_df[pred_df["doc_id"] == doc_id][["entity_text", "label"]].reset_index(drop=True)
    gold = gold_df[gold_df["doc_id"] == doc_id][["entity_text", "label"]].reset_index(drop=True)
    return text, pred, gold


sample_doc_id = docs_df.loc[0, "doc_id"]
text, pred_entities, gold_entities = review_document(sample_doc_id)
print("TEXT:\n", text)
print("\nPREDICTED:")
display(pred_entities)
print("GOLD:")
display(gold_entities)

TEXT:
 On 2026-01-12, Liam Walker from Fabrikam Ops signed a contract in Lisbon for $18,900.

PREDICTED:


,entity_text,label
0,2026-01-12,DATE
1,Liam Walker,PERSON
2,Fabrikam Ops,ORG
3,Lisbon,GPE
4,"18,900",MONEY


GOLD:


,entity_text,label
0,2026-01-12,DATE
1,Liam Walker,PERSON
2,Fabrikam Ops,ORG
3,Lisbon,GPE
4,"$18,900",MONEY


## 9. Summary

- Data is externalized under `data/04_data` for teaching and repeatability.
- The notebook covers extraction + measurable evaluation (not only entity printing).
- This workflow is suitable for enterprise extraction pilots and model monitoring baselines.
